# AI Training Studio — Google Colab Launcher (Phase 1 MVP)

Notebook นี้ใช้สำหรับติดตั้ง dependencies และเปิดหน้าเว็บ **AI Training Studio (Gradio)** สำหรับเทรน **Ultralytics YOLO11 Object Detection** ครบวงจรโดยไม่ต้องแก้ไขโค้ด Python

### ⚠️ ข้อสำคัญเรื่อง Colab Temporary Storage
- ระบบจัดเก็บ Dataset, Checkpoints (`best.pt`, `last.pt`) และไฟล์ Export ไว้ในดิสก์ชั่วคราวของ Colab Runtime (`/content/ai_training_workspace`)
- **หาก Browser หลุดแต่ Runtime ยังมีชีวิต:** งานเทรนยังรันต่อเบื้องหลัง และคุณสามารถเปิดลิงก์เดิมแล้วกด **Reconnect** ได้โดยไม่เริ่มเทรนซ้ำ
- **หาก Colab Runtime ถูกยุติ (Timeout / Disconnect Runtime / ครบโควต้า):** ไฟล์ทั้งหมดใน Runtime จะหายและไม่สามารถกู้คืนได้ กรุณาดาวน์โหลด `best.pt` / `last.pt` ลงเครื่องจากหน้า **Monitor / Sessions / Export**

In [ ]:
# 1. ตรวจสอบ GPU และสภาพแวดล้อมของ Google Colab
import os, platform, subprocess, sys

print('Python:', platform.python_version())
res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if res.returncode == 0:
    print(res.stdout)
else:
    print('⚠️ ไม่พบ GPU — แนะนำไปที่ Runtime -> Change runtime type -> เลือก T4 / L4 GPU')

In [ ]:
# 2. ตั้งค่าตำแหน่งโปรเจกต์ (แก้ REPO_URL หากโหลดจาก GitHub หรือใช้โฟลเดอร์ที่อัปโหลดไว้แล้ว)
import os
from pathlib import Path

REPO_URL = ''  # ใส่ URL ของ Git repo เช่น 'https://github.com/<user>/<repo>.git' หรือเว้นว่างหากโค้ดอยู่ใน /content/AI_Colab แล้ว
PROJECT_DIR = Path('/content/AI_Colab')
WORKSPACE_DIR = Path('/content/ai_training_workspace')

if REPO_URL and not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
elif not PROJECT_DIR.exists() and Path('app.py').exists():
    PROJECT_DIR = Path.cwd()

assert (PROJECT_DIR / 'app.py').exists(), (
    f'ไม่พบ app.py ที่ {PROJECT_DIR} กรุณาระบุ REPO_URL หรืออัปโหลดโฟลเดอร์โปรเจกต์ไปที่ /content/AI_Colab'
)
os.chdir(PROJECT_DIR)
os.environ['AI_STUDIO_WORKSPACE'] = str(WORKSPACE_DIR)
print('Project Dir  :', PROJECT_DIR)
print('Workspace Dir:', WORKSPACE_DIR)

In [ ]:
# 3. ติดตั้ง Dependencies ที่กำหนดเวอร์ชันเข้ากันได้ (ไม่ทับ PyTorch CUDA ของ Colab)
!pip install -q -r requirements.txt

# พิมพ์เวอร์ชันแพ็กเกจทั้งหมดก่อนเปิดแอป
from pathlib import Path
from infrastructure.monitoring.system_monitor import format_startup_banner
print(format_startup_banner(Path(os.environ['AI_STUDIO_WORKSPACE'])))

In [ ]:
# 4. เปิด AI Training Studio (Gradio Web UI)
# Training จะรันใน Background Subprocess แยกจาก UI เพื่อให้ Reconnect ได้หากหน้าเว็บหลุด
!python app.py --share --workspace /content/ai_training_workspace